# On time, in full, and wrong: the shipping records

The runnable companion to [On time, in full, and wrong](https://kvinayakpai.github.io/domain-series/logistics/) and its
[technical companion](https://kvinayakpai.github.io/domain-series/logistics/technical/). Everything here uses the downloadable records of Store 014 and
the warehouse that supplies it: fictional, simulated hour by hour through June 2026. The answer key is read only at
the end of each section, to score what the records gave us.

Runs in a few seconds. Needs `duckdb` and `pandas` (both preinstalled in Colab).

In [1]:
import json, os, urllib.request

import duckdb
import pandas as pd

DATA = os.environ.get("STORE014_DATA", "https://kvinayakpai.github.io/domain-series/data/india-grocery/")
FILES = {
    "products": "products.csv", "shipments": "shipments-june.csv", "notice_lines": "shipping-notice-lines-june.csv",
    "picks": "dc-pick-confirmations-june.csv", "pallets": "pallets-june.csv", "goods_receipts": "goods-receipts-june.csv",
    "dc_adjustments": "dc-stock-adjustments-june.csv", "stock_counts": "stock-counts-june.csv",
    "store_orders": "store-orders-june.csv", "inventory_book": "inventory-book-june.csv",
    "daily_sales": "daily-sales-june.csv", "forecasts": "forecasts-june.csv",
}
TIMES = {"shipments": ["dispatched_ts", "slot_start", "slot_end", "arrived_ts"], "picks": ["pick_ts"],
         "goods_receipts": ["received_ts"], "dc_adjustments": ["adjusted_ts"], "stock_counts": ["count_ts"],
         "store_orders": ["created_ts"]}
TEXT = {"sscc": str, "gtin": str}  # SSCCs and GTINs are identifiers: keep their leading zeros

con = duckdb.connect()
tables = {}
for name, file in FILES.items():
    df = pd.read_csv(DATA + file, parse_dates=TIMES.get(name, []), dtype=TEXT)
    tables[name] = df
    con.register(name, df)

def read_json(path):
    if path.startswith("http"):
        with urllib.request.urlopen(path) as f:
            return json.load(f)
    with open(path) as f:
        return json.load(f)

KEY = read_json(DATA + "answer-key-logistics.json")   # for scoring only

def q(sql):
    return con.execute(sql).df()

{k: len(v) for k, v in tables.items()}

{'products': 11,
 'shipments': 25,
 'notice_lines': 70,
 'picks': 70,
 'pallets': 30,
 'goods_receipts': 70,
 'dc_adjustments': 2,
 'stock_counts': 25,
 'store_orders': 68,
 'inventory_book': 308,
 'daily_sales': 308,
 'forecasts': 196}

## 1. Tests that pass

Ten data tests on the shipping records, the kind a dbt project ships on day one. A test passes when its query returns
no rows. Every record here was written from the same pick confirmation, so they agree with each other.

In [2]:
TESTS = {
 "Shipments: arrival after dispatch, a 30-minute slot":
   "SELECT * FROM shipments WHERE arrived_ts <= dispatched_ts OR slot_end - slot_start <> INTERVAL 30 MINUTE",
 "Shipping notice: shipment, pallet and line are unique":
   "SELECT shipment_id, sscc, line, count(*) FROM notice_lines GROUP BY ALL HAVING count(*) > 1",
 "Shipping notice: items and barcodes in the product master, units are cases times the case size": """
   SELECT n.* FROM notice_lines n LEFT JOIN products p ON p.sku = n.sku AND p.gtin = n.gtin
   WHERE p.sku IS NULL OR n.units <> n.cases * p.case_units""",
 "Pick confirmations: one per notice line, same order, item and cases, confirmed by a scan": """
   SELECT n.* FROM notice_lines n LEFT JOIN picks p USING (shipment_id, line)
   WHERE p.sku IS NULL OR p.sku <> n.sku OR p.order_id <> n.order_id OR p.cases_confirmed <> n.cases
      OR p.confirmed_by IS NULL OR p.cases_confirmed > p.cases_ordered""",
 "Pallets: every pallet on a notice is in the pallet table, with the same tare on every pallet": """
   WITH w AS (SELECT n.sscc, sum(n.cases * p.case_kg) AS cases_kg FROM notice_lines n JOIN products p USING (sku) GROUP BY n.sscc),
        t AS (SELECT round(pl.notice_kg - w.cases_kg, 1) AS tare FROM w LEFT JOIN pallets pl USING (sscc))
   SELECT tare, count(*) FROM t GROUP BY tare HAVING (SELECT count(DISTINCT tare) FROM t) <> 1 OR tare IS NULL""",
 "Goods receipts: every notice line booked, as its shipping notice says": """
   SELECT n.* FROM notice_lines n LEFT JOIN goods_receipts r USING (shipment_id, sscc, line)
   WHERE r.grn_id IS NULL OR r.asn_sku <> n.sku OR r.asn_units <> n.units OR r.sku_booked <> r.asn_sku
      OR r.units_booked <> r.asn_units""",
 "Goods receipts: booked when the truck arrived":
   "SELECT r.* FROM goods_receipts r JOIN shipments s USING (shipment_id) WHERE r.received_ts <> s.arrived_ts",
 "Store orders: every order line is on the next morning's shipping notice": """
   SELECT o.* FROM store_orders o LEFT JOIN notice_lines n ON n.order_id = o.order_id AND n.sku = o.sku
   WHERE n.sku IS NULL AND CAST(o.created_ts AS DATE) < (SELECT max(CAST(arrived_ts AS DATE)) FROM shipments)""",
 "Warehouse adjustments: units are cases times the case size, value is units at cost": """
   SELECT a.* FROM dc_adjustments a JOIN products p USING (sku)
   WHERE a.units <> a.cases * p.case_units OR abs(a.value_at_cost_inr - a.units * p.cost_inr) > 0.01""",
 "Store counts: each adjustment is the count minus the book":
   "SELECT * FROM stock_counts WHERE adjustment <> counted - book_before",
}
results = {name: len(q(sql)) for name, sql in TESTS.items()}
pd.Series(results, name="failing rows")

Shipments: arrival after dispatch, a 30-minute slot                                               0
Shipping notice: shipment, pallet and line are unique                                             0
Shipping notice: items and barcodes in the product master, units are cases times the case size    0
Pick confirmations: one per notice line, same order, item and cases, confirmed by a scan          0
Pallets: every pallet on a notice is in the pallet table, with the same tare on every pallet      0
Goods receipts: every notice line booked, as its shipping notice says                             0
Goods receipts: booked when the truck arrived                                                     0
Store orders: every order line is on the next morning's shipping notice                           0
Warehouse adjustments: units are cases times the case size, value is units at cost                0
Store counts: each adjustment is the count minus the book                                         0


## 2. Corrections, in both buildings

The warehouse corrects its slots; the store corrects its books at counts. Put every correction in one view, in units
and in cases, because a mis-pick moves whole cases and the two sizes of atta have different case sizes.

In [3]:
con.execute("""
CREATE OR REPLACE VIEW v_corrections AS
SELECT 'warehouse' AS building, a.adjusted_ts AS ts, a.sku, split_part(a.sku, '-', 1) AS family, a.slot,
       a.units, CAST(a.cases AS DOUBLE) AS cases, a.value_at_cost_inr AS value_inr, a.reason AS why
FROM dc_adjustments a
UNION ALL
SELECT 'store', c.count_ts, c.sku, split_part(c.sku, '-', 1), NULL, c.adjustment,
       c.adjustment / CAST(p.case_units AS DOUBLE), round(c.adjustment * p.cost_inr, 2), c.trigger
FROM stock_counts c JOIN products p USING (sku)
WHERE c.adjustment <> 0 AND c.count_ts > TIMESTAMP '2026-06-01 07:00'
""")
q("SELECT * FROM v_corrections ORDER BY ts")

,building,ts,sku,family,slot,units,cases,value_inr,why
0,warehouse,2026-06-23 14:20:00,ATTA-WW-1KG,ATTA,A-07-04,-160,-16.000000,-7920.0,short pick: slot count variance
1,warehouse,2026-06-25 10:00:00,ATTA-WW-5KG,ATTA,A-07-03,64,16.000000,14592.0,cycle count variance
2,store,2026-06-26 09:00:00,ATTA-WW-10KG,ATTA,NaN,-1,-0.500000,-431.0,manager request after complaints
3,store,2026-06-26 09:00:00,ATTA-WW-5KG,ATTA,NaN,-63,-15.750000,-14364.0,manager request after complaints
4,store,2026-06-26 09:00:00,ATTA-WW-1KG,ATTA,NaN,157,15.700000,7771.5,manager request after complaints
5,store,2026-06-28 22:30:00,ATTA-WW-5KG,ATTA,NaN,-1,-0.250000,-228.0,scheduled full count
6,store,2026-06-28 22:30:00,ATTA-WW-1KG,ATTA,NaN,1,0.100000,49.5,scheduled full count
7,store,2026-06-28 22:30:00,RICE-SM-5KG,RICE,NaN,-2,-0.500000,-602.0,scheduled full count
8,store,2026-06-28 22:30:00,OIL-SUN-1L,OIL,NaN,-8,-0.800000,-1072.0,scheduled full count
9,store,2026-06-28 22:30:00,SUGAR-1KG,SUGAR,NaN,-1,-0.100000,-42.0,scheduled full count


Netted, the atta corrections come to almost nothing. That is why nobody chases them.

In [4]:
q("""SELECT round(sum(value_inr)) AS net_inr, round(sum(abs(value_inr))) AS gross_inr
     FROM v_corrections WHERE family = 'ATTA' AND abs(cases) >= 1""")

,net_inr,gross_inr
0,80.0,44648.0


## 3. Mirror pairs

The domain test: two corrections mirror each other if they are in one product family, have opposite signs, are each
at least a case, are within 15% of each other in cases, and fall within seven days. In one building they must be in
neighbouring slots (warehouse) or two items at the store; across buildings they must be the same item.

In [5]:
con.execute("""
CREATE OR REPLACE VIEW v_mirror_pairs AS
SELECT CASE WHEN a.building = b.building THEN 'same building' ELSE 'across buildings' END AS kind,
       a.building AS building_a, a.ts AS ts_a, a.sku AS sku_a, a.slot AS slot_a, a.units AS units_a, round(a.cases, 2) AS cases_a,
       b.building AS building_b, b.ts AS ts_b, b.sku AS sku_b, b.slot AS slot_b, b.units AS units_b, round(b.cases, 2) AS cases_b
FROM v_corrections a JOIN v_corrections b
  ON a.family = b.family AND sign(a.units) = -sign(b.units)
 AND abs(a.cases) >= 1 AND abs(b.cases) >= 1
 AND abs(abs(a.cases) - abs(b.cases)) <= 0.15 * greatest(abs(a.cases), abs(b.cases))
 AND abs(epoch(b.ts) - epoch(a.ts)) <= 7 * 86400
 AND (a.ts < b.ts OR (a.ts = b.ts AND a.sku < b.sku))
WHERE (a.building = b.building AND a.sku <> b.sku
       AND (a.building = 'store' OR (split_part(a.slot, '-', 1) = split_part(b.slot, '-', 1)
            AND split_part(a.slot, '-', 2) = split_part(b.slot, '-', 2)
            AND abs(CAST(split_part(a.slot, '-', 3) AS INTEGER) - CAST(split_part(b.slot, '-', 3) AS INTEGER)) = 1)))
   OR (a.building <> b.building AND a.sku = b.sku)
""")
pairs = q("SELECT * FROM v_mirror_pairs ORDER BY ts_b, ts_a")
pairs

,kind,building_a,ts_a,sku_a,slot_a,units_a,cases_a,building_b,ts_b,sku_b,slot_b,units_b,cases_b
0,same building,warehouse,2026-06-23 14:20:00,ATTA-WW-1KG,A-07-04,-160,-16.0,warehouse,2026-06-25 10:00:00,ATTA-WW-5KG,A-07-03,64,16.00
1,across buildings,warehouse,2026-06-23 14:20:00,ATTA-WW-1KG,A-07-04,-160,-16.0,store,2026-06-26 09:00:00,ATTA-WW-1KG,NaN,157,15.70
2,across buildings,warehouse,2026-06-25 10:00:00,ATTA-WW-5KG,A-07-03,64,16.0,store,2026-06-26 09:00:00,ATTA-WW-5KG,NaN,-63,-15.75
3,same building,store,2026-06-26 09:00:00,ATTA-WW-1KG,NaN,157,15.7,store,2026-06-26 09:00:00,ATTA-WW-5KG,NaN,-63,-15.75
4,same building,store,2026-06-28 22:30:00,MILK-FC-500ML,NaN,48,2.0,store,2026-06-28 22:30:00,MILK-TONED-500ML,NaN,-55,-2.29


Four atta corrections, two in each building, make four pairs. The fifth pair is milk at the month-end count: toned
milk scanned as full-cream at the till, the retail post's scan errors.

## 4. The trace

For the warehouse pair, look for picks of the item that came up over (5 kg, slot A-07-03), in the cases the pair moved,
in the week before the first correction. A slot-label scan confirmed each of them.

In [6]:
dc = pairs[(pairs.kind == "same building") & (pairs.building_a == "warehouse")].iloc[0]
sku, slot, cases = (dc.sku_a, dc.slot_a, round(abs(dc.cases_a))) if dc.units_a > 0 else (dc.sku_b, dc.slot_b, round(abs(dc.cases_b)))
first = min(dc.ts_a, dc.ts_b)
candidates = q(f"""
  SELECT p.pick_ts, p.shipment_id, p.line, p.sku, p.slot, p.cases_confirmed, p.confirmed_by, n.sscc, n.units
  FROM picks p JOIN notice_lines n USING (shipment_id, line)
  WHERE p.sku = '{sku}' AND p.slot = '{slot}' AND p.cases_confirmed = {cases}
    AND p.pick_ts BETWEEN TIMESTAMP '{first}' - INTERVAL 7 DAY AND TIMESTAMP '{first}'
""")
candidates

,pick_ts,shipment_id,line,sku,slot,cases_confirmed,confirmed_by,sscc,units
0,2026-06-22 03:00:00,SHP-260622,1,ATTA-WW-5KG,A-07-03,16,slot label scan,029000010000010542,64


Score it: what was really on that pallet line?

In [7]:
c0 = candidates.iloc[0]
truth = [c for c in KEY["pallet_contents_june"] if c["sscc"] == c0.sscc and c["line"] == c0.line][0]
{"noticed": f"{c0.units} x {c0.sku}", "really": f"{truth['units_actual']} x {truth['sku_actual']}"}

{'noticed': '64 x ATTA-WW-5KG', 'really': '160 x ATTA-WW-1KG'}

## 5. Receipts that do not survive the next count

A receipt booked by a pallet-label scan copies the shipping notice. A later count is an independent observation. Flag
the latest receipt before a count when the count takes off half or more of what that receipt booked.

In [8]:
con.execute("""
CREATE OR REPLACE VIEW v_receipt_survival AS
WITH latest AS (
  SELECT c.count_ts, c.sku, c.book_before, c.counted, c.adjustment, max(r.received_ts) AS received_ts
  FROM stock_counts c JOIN goods_receipts r ON r.sku_booked = c.sku AND r.received_ts < c.count_ts
   AND r.received_ts >= c.count_ts - INTERVAL 7 DAY
  WHERE c.adjustment < 0 GROUP BY ALL)
SELECT r.shipment_id, r.sscc, r.line, r.received_ts, r.sku_booked AS sku, r.units_booked,
       l.count_ts, l.book_before, l.counted, l.adjustment
FROM latest l JOIN goods_receipts r ON r.sku_booked = l.sku AND r.received_ts = l.received_ts
WHERE -l.adjustment >= 0.5 * r.units_booked
""")
survival = q("SELECT * FROM v_receipt_survival")
survival

,shipment_id,sscc,line,received_ts,sku,units_booked,count_ts,book_before,counted,adjustment
0,SHP-260622,029000010000010542,1,2026-06-22 06:12:00,ATTA-WW-5KG,64,2026-06-26 09:00:00,63,0,-63


## 6. OTIF, four ways

By the warehouse's picks, by the store's receipts per line and per delivery, and once more with the receipts that did
not survive a count taken out. Then compare with the answer key, which knows what was on the pallets.

In [9]:
per = q("""
  WITH lines AS (
    SELECT n.shipment_id, n.line, p.cases_ordered * pr.case_units AS ordered, r.units_booked AS received,
           p.cases_confirmed, p.cases_ordered, s.arrived_ts BETWEEN s.slot_start AND s.slot_end AS in_slot,
           strftime(s.dispatched_ts, '%H:%M') = '04:30' AS dispatched_on_time
    FROM notice_lines n JOIN shipments s USING (shipment_id) JOIN picks p USING (shipment_id, line)
    JOIN products pr ON pr.sku = n.sku JOIN goods_receipts r USING (shipment_id, sscc, line))
  SELECT shipment_id, count(*) AS lines, bool_and(in_slot) AS in_slot,
         bool_and(dispatched_on_time AND cases_confirmed = cases_ordered) AS dc_report,
         bool_and(in_slot AND received = ordered) AS scorecard,
         sum(CASE WHEN in_slot AND received = ordered THEN 1 ELSE 0 END) AS lines_ok
  FROM lines GROUP BY shipment_id ORDER BY shipment_id""")
bad = set(survival.shipment_id)
per["reconciled"] = per.scorecard & ~per.shipment_id.isin(bad)
ways = {"dc_report": per.dc_report.mean(), "by_line": per.lines_ok.sum() / per.lines.sum(),
        "scorecard": per.scorecard.mean(), "reconciled": per.reconciled.mean()}
pd.DataFrame({"from the records": pd.Series(ways).round(3),
              "answer key": pd.Series(KEY["otif_four_ways"]["ways"]).rename({"at_the_shelf": "reconciled"})})

,from the records,answer key
dc_report,0.92,0.92
by_line,0.90,0.90
scorecard,0.88,0.88
reconciled,0.84,0.84


## Exercises

1. Find the 22 June line from the pallet weights alone. The pallet table holds the weight each pallet's notice implies;
   what would a scale have read?
2. Loosen the mirror rule to 25% and to 30 days. What else pairs up, and is any of it real?
3. Build the pair rule for the store alone: which store counts could have told the store, on their own, that a
   delivery was wrong?
4. Compute OTIF per line and per delivery for a 15-minute slot instead of 30. Which deliveries move?

The answer key (`answer-key-logistics.json`) scores each one.

In [10]:
# The numbers the post quotes, in one place.
summary = {
    "generic_tests_passed": int(sum(v == 0 for v in results.values())),
    "pairs": len(pairs), "atta_pairs": int(pairs.sku_a.str.startswith("ATTA").sum()),
    "trace": [c0.shipment_id, c0.sscc, int(c0.line)],
    "receipts_failing": len(survival),
    "otif": {k: round(float(v), 3) for k, v in ways.items()},
}
print(json.dumps(summary))

{"generic_tests_passed": 10, "pairs": 5, "atta_pairs": 4, "trace": ["SHP-260622", "029000010000010542", 1], "receipts_failing": 1, "otif": {"dc_report": 0.92, "by_line": 0.9, "scorecard": 0.88, "reconciled": 0.84}}
